In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

data = pd.read_csv("horse_colic_normalized.csv")

df_reduced = pd.read_csv(
    "horse_colic_reduced_encoded.csv"
)

target_col = "outcome"

data = data.dropna(subset=[target_col])
df_reduced = df_reduced.dropna(subset=[target_col])

print(f"Размер датасета: {data.shape[0]} строк, {data.shape[1]} столбцов")
display(data.head())

print(f"Размер сокращённого датасета: {df_reduced.shape[0]} строк, {df_reduced.shape[1]} столбцов")
display(df_reduced.head())

Размер датасета: 366 строк, 28 столбцов


,surgery,age,hospital_number,rectal_temperature,pulse,respiratory_rate,temperature_of_extremities,peripheral_pulse,mucous_membranes,capillary_refill_time,...,packed_cell_volume,total_protein,abdominocentesis_appearance,abdominocentesis_total_protein,outcome,lesion_site,lesion_type,lesion_subtype,cp_data,surgical_lesion
0,2.0,1,530101,38.5,66.0,28.0,3.0,3.0,NaN,2.0,...,45.0,8.4,NaN,NaN,2.0,11300,0,0,2,2
1,1.0,1,534817,39.2,88.0,20.0,NaN,NaN,4.0,1.0,...,50.0,85.0,2.0,2.0,3.0,2208,0,0,2,2
2,2.0,1,530334,38.3,40.0,24.0,1.0,1.0,3.0,1.0,...,33.0,6.7,NaN,NaN,1.0,0,0,0,1,2
3,1.0,9,5290409,39.1,164.0,84.0,4.0,1.0,6.0,2.0,...,48.0,7.2,3.0,5.3,2.0,2208,0,0,1,1
4,2.0,1,530255,37.3,104.0,35.0,NaN,NaN,6.0,2.0,...,74.0,7.4,NaN,NaN,2.0,4300,0,0,2,2


Размер сокращённого датасета: 366 строк, 15 столбцов


,surgery,hospital_number,rectal_temperature,pulse,respiratory_rate,mucous_membranes,pain,peristalsis,abdominal_distension,abdomen,packed_cell_volume,total_protein,outcome,lesion_site,surgical_lesion
0,2.0,530101,38.5,66.0,28.0,NaN,5.0,4.0,4.0,5.0,45.0,8.4,2.0,11300,2
1,1.0,534817,39.2,88.0,20.0,4.0,3.0,4.0,2.0,2.0,50.0,85.0,3.0,2208,2
2,2.0,530334,38.3,40.0,24.0,3.0,3.0,3.0,1.0,1.0,33.0,6.7,1.0,0,2
3,1.0,5290409,39.1,164.0,84.0,6.0,2.0,4.0,4.0,NaN,48.0,7.2,2.0,2208,1
4,2.0,530255,37.3,104.0,35.0,6.0,NaN,NaN,NaN,NaN,74.0,7.4,2.0,4300,2


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns

X = data.drop(target_col, axis=1)
y = data[target_col]

X = X.fillna(X.median())

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

clf = RandomForestClassifier(n_estimators=2000, random_state=42)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)

print("Качество модели:")
print(classification_report(y_test, y_pred, zero_division=0))


X_reduced = df_reduced.drop(target_col, axis=1)
y_reduced = df_reduced[target_col]

X_reduced = X_reduced.fillna(X_reduced.median())

X_train_reduced, X_test_reduced, y_train_reduced, y_test_reduced = train_test_split(
    X_reduced,
    y_reduced,
    test_size=0.3,
    random_state=42,
    stratify=y_reduced
)

clf_reduced = RandomForestClassifier(n_estimators=2000, random_state=42)
clf_reduced.fit(X_train_reduced, y_train_reduced)

y_pred_reduced = clf_reduced.predict(X_test_reduced)

print("Качество модели на сокращённом датасете:")
print(classification_report(y_test_reduced, y_pred_reduced, zero_division=0))

Качество модели:
              precision    recall  f1-score   support

         1.0       0.79      0.91      0.85        67
         2.0       0.72      0.78      0.75        27
         3.0       0.75      0.19      0.30        16

    accuracy                           0.77       110
   macro avg       0.76      0.63      0.63       110
weighted avg       0.77      0.77      0.74       110



In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)

sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues')
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()


conf_matrix_reduced = confusion_matrix(y_test_reduced, y_pred_reduced)

sns.heatmap(
    conf_matrix_reduced,
    annot=True,
    fmt='d',
    cmap='Blues'
)

plt.title('Confusion Matrix для сокращённого датасета')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

positive_class = 1

y_test_binary = (y_test == positive_class).astype(int)

class_index = list(clf.classes_).index(positive_class)
y_pred_prob = clf.predict_proba(X_test)[:, class_index]

fpr, tpr, _ = roc_curve(y_test_binary, y_pred_prob)
auc = roc_auc_score(y_test_binary, y_pred_prob)

plt.plot(fpr, tpr, label=f'Class {positive_class} vs rest, AUC = {auc:.2f}')
plt.plot([0, 1], [0, 1], 'k--')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC-кривая RandomForest')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn import tree
import graphviz

estimator = clf.estimators_[0]

dot_data = tree.export_graphviz(
    estimator,
    out_file=None,
    feature_names=X.columns,
    class_names=[str(i) for i in clf.classes_],
    filled=True,
    rounded=True,
    special_characters=True
)

graph = graphviz.Source(dot_data)

output_path = 'tree_visualization_horse_colic.png'

graph.render(output_path.replace('.png', ''), format='png', cleanup=True)
print(f"Файл сохранен: {output_path}")

estimator_reduced = clf_reduced.estimators_[0]

dot_data_reduced = tree.export_graphviz(
    estimator_reduced,
    out_file=None,
    feature_names=X_reduced.columns,
    class_names=[str(i) for i in clf_reduced.classes_],
    filled=True,
    rounded=True,
    special_characters=True
)

graph_reduced = graphviz.Source(dot_data_reduced)

output_path_reduced = 'tree_visualization_horse_colic_reduced.png'

graph_reduced.render(
    output_path_reduced.replace('.png', ''),
    format='png',
    cleanup=True
)

print(f"Файл сохранен: {output_path_reduced}")